# MindEye2 · category experiment (subject 1, 700 animal-only images)

Runtime → Change runtime type → **T4 GPU**. Run the cells **top to bottom**.

**After every reconnect:** run cells 1 and 2 again, then continue from the first stage
that is not finished yet. Finished stages are saved on your Google Drive
(`MyDrive/mindeye2_lora`) and are not redone. Cell 4 shows what is finished.

## 1 · Connect Google Drive (always first)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2 · Get / update the code and install packages

In [ ]:
import os
REPO = "/content/MindEye2_category"
if not os.path.isdir(f"{REPO}/.git"):
    !git clone https://github.com/AminMDNZ/MindEye2_category.git {REPO}
%cd {REPO}
!git pull
!bash setup/colab_setup.sh
C = "--config configs/category_animal.yaml"

## 3 · Tests (expect: all passed)

In [ ]:
!pytest tests/ -q

## 4 · Where the work is saved, and what is finished

`free space` must be in **TB** (your Drive). About 100 GB means Colab's temporary disk: Drive is not connected, so stop and run cell 1.

In [ ]:
!python -m mindeye_lora.cli status $C

## 5 · Labels and data (once)

`categories`: COCO labels + subject 1's trial table (~600 MB download, a few minutes).
`assets`: picks the 700 animal-only images and fetches their brain data and images (~15 min).
Both finish in seconds once done.

In [ ]:
!python -m mindeye_lora.cli categories $C

In [ ]:
!python -m mindeye_lora.cli assets $C

## 6 · CLIP targets (~5 min)

In [ ]:
!python -m mindeye_lora.cli precompute $C

## 7 · LoRA fine-tuning (~75 min on a T4)

If Colab disconnects: run cells 1 and 2, then this cell again. It resumes from the last checkpoint.

In [ ]:
!python -m mindeye_lora.cli train $C

## 8 · Predict and score the 1,000 test images

In [ ]:
!python -m mindeye_lora.cli predict $C

In [ ]:
!python -m mindeye_lora.cli evaluate $C

## 9 · Compare with the session-1 run (step 3)

Same arm and seed, same test images: the 179 animal-only test images, the other 821, all 1,000, plus animal-vs-animal discrimination. Reads saved scores and predictions; trains nothing (~2 min).

In [ ]:
!python -m mindeye_lora.cli compare-category $C

## 10 · Reconstruct the animal test images from both runs

SDXL unCLIP decoder, ~30 s per image on a T4: 179 images x 2 runs is about 3 hours. The first time it also downloads the decoder (18 GB, stored on Drive as ~9 GB of shards). If Colab disconnects, run cells 1 and 2, then this cell again WITHOUT --force: it resumes. Keep the runtime open ~2 minutes after it finishes, so Drive can save the files.

Add `--n_images 24` for a quick look first.

In [ ]:
!python -m mindeye_lora.cli recon-category $C

In [ ]:
from IPython.display import Image
Image('/content/drive/MyDrive/mindeye2_lora/results/cat-animal-only-700-first-s0/figures/category_reconstructions.png')